# Virtual Asset Collections — STAC

End-to-end loop demonstrating how the `/virtual/assets/...` STAC endpoints
expose a single uploaded asset across every collection it belongs to.

1. Create a catalog
2. Create two collections in that catalog
3. Register the **same `asset_id`** in both collections
4. **NEW** — `GET /stac/virtual/assets/{asset_id}/catalogs/{cat}/collections`
   lists every collection in the catalog where that asset has rows
5. Drill into one of the child links
   (`/virtual/assets/{asset_id}/catalogs/{cat}/collections/{coll}`)
6. Cleanup

The plural `/collections` route was missing before geoid v0.6.23 — only
`/.../collections/{collection_id}` (singular) existed, so the natural
back-navigation produced a 404. The new handler closes the loop:
given an asset, you can list every collection that references it without
knowing the collection ids ahead of time.

In [ ]:
import os
import httpx

BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"
CATALOG_ID = "demo-virtual-assets"
COLL_A = "sensors-north"
COLL_B = "sensors-south"
ASSET_ID = "shared-station-batch-2026-04"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

def _check(r, label=""):
    status = r.status_code
    suffix = "" if status < 400 else f" \u2014 {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

## 1. Create the catalog

Idempotent against re-runs of the notebook \u2014 a 409 on existing catalog is
treated as success.

In [ ]:
r = await client.post(
    "/stac/catalogs",
    json={
        "id": CATALOG_ID,
        "title": "Virtual Asset Demo",
        "description": "Catalog for the virtual-asset-collections walkthrough.",
    },
)
if r.status_code == 409:
    print(f"Catalog: 409 (already exists, OK)")
else:
    _check(r, "Catalog")

## 2. Create two collections

Both use the default routing config (Postgres primary).  No schema or
write policy needed for this walkthrough; the waterfall fills every
sub-config from the code defaults.

In [ ]:
WORLD_EXTENT = {
    "spatial": {"bbox": [[-180, -90, 180, 90]]},
    "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
}

for coll in (COLL_A, COLL_B):
    r = await client.post(
        f"/stac/catalogs/{CATALOG_ID}/collections",
        json={
            "id": coll,
            "title": coll.replace("-", " ").title(),
            "description": f"{coll} demo collection.",
            "license": "proprietary",
            "extent": WORLD_EXTENT,
        },
    )
    if r.status_code == 409:
        print(f"Collection {coll}: 409 (already exists, OK)")
    else:
        _check(r, f"Collection {coll}")

## 3. Register the same asset in both collections

The virtual-collections endpoint reads from the per-tenant `assets`
table.  An asset row is keyed by `(asset_id, catalog_id, collection_id)`
\u2014 the same `asset_id` can therefore appear once per collection.

We POST the same `asset_id` into each collection so the new endpoint
has two children to surface.

In [ ]:
asset_payload = {
    "asset_id": ASSET_ID,
    "uri": "https://example.com/data/shared-station-batch-2026-04.csv",
    "asset_type": "ASSET",
    "metadata": {
        "description": "Shared sensor batch ingested into multiple collections.",
        "format": "text/csv",
    },
}

for coll in (COLL_A, COLL_B):
    r = await client.post(
        f"/assets/catalogs/{CATALOG_ID}/collections/{coll}",
        json=asset_payload,
    )
    if r.status_code == 409:
        print(f"Asset in {coll}: 409 (already exists, OK)")
    else:
        _check(r, f"Asset in {coll}")

## 4. List all collections in the catalog containing the asset

**`GET /stac/virtual/assets/{asset_id}/catalogs/{cat}/collections`** \u2014 the
endpoint added in geoid v0.6.23.  Returns a `pystac.Catalog`-shaped
envelope with one `child` link per collection where the asset has rows.

Each child link points at the existing
`/.../collections/{collection_id}` virtual-collection endpoint, where
the items view is filtered by `asset_id`.

In [ ]:
r = await client.get(
    f"/stac/virtual/assets/{ASSET_ID}/catalogs/{CATALOG_ID}/collections",
)
_check(r, "Virtual collections list")
doc = r.json()
print(f"id    = {doc.get('id')}")
print(f"title = {doc.get('title')}")
print("children:")
for link in doc.get("links", []):
    if link.get("rel") == "child":
        print(f"  - {link.get('title')}: {link.get('href')}")

## 5. Drill into one of the children

Each child resolves to the single-asset virtual-collection view
(`get_virtual_asset_collection`).  Its `items` link drills further into
`get_virtual_asset_items` which filters the original collection by
`asset_id` \u2014 so even if the collection holds millions of items, the
asset-scoped view returns only those linked to this `asset_id`.

In [ ]:
child_href = next(
    (l["href"] for l in doc.get("links", []) if l.get("rel") == "child"),
    None,
)
assert child_href, "expected at least one child link in the virtual list"

# Strip the BASE prefix so httpx can route the relative path.
rel_path = child_href.split(BASE, 1)[1] if BASE in child_href else child_href
r = await client.get(rel_path)
_check(r, "Virtual single collection")
print("keys:", sorted(r.json().keys())[:10])

## 6. Cleanup

`?force=true` cascades through collections + assets so a re-run of the
notebook starts from a clean slate.

In [ ]:
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
print(f"Cleanup: {r.status_code}")
if r.status_code >= 400:
    print(f"  (best-effort; manual cleanup may be required) {r.text[:200]}")
await client.aclose()